# Flagship Option B: Hybrid State Rollback Kill Experiment
**Paper:** Same-Subspace State Overwrite & Synchronized State-KV Rollback in Hybrid Recurrent Reasoning Models
**Target Venue:** NeurIPS 2027 Flagship Submission
**Lead Researcher:** Jaime Jarauta

### One-Click Execution
Press **Runtime -> Run all** (`Cmd+F9` / `Ctrl+F9`).
This notebook self-heals, downloads `Qwen/Qwen3.5-0.8B`, runs the full 100-item MATH-500 backtrack evaluation across all 7 arms, generates paired-bootstrap 95% CIs, and automatically unassigns the GPU when finished.

In [ ]:
# Step 1: Self-Healing Environment Setup & Credit Guard
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print('\n[Credit Guard] Unhandled error or interrupt! Unassigning Colab GPU...')
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), lambda self, et, val, tb, tb_offset=None: (self.showtraceback((et, val, tb), tb_offset=tb_offset), auto_teardown()))

print('[Ready] Credit guard active. VM will self-terminate on error or completion.')


In [ ]:
# Step 2: Clone Repository & Install Dependencies
import os, sys, subprocess

repo_url = 'https://github.com/jaimejarauta/decoupled-delta-net.git'
branch = 'feature/hybrid-state-rollback'
target_dir = '/content/hybrid-state-rollback'

%cd /content
if not os.path.exists(target_dir):
    print(f'Cloning hybrid-state-rollback from {repo_url} (branch: {branch})...')
    subprocess.run(['git', 'clone', '-b', branch, repo_url, target_dir], check=True)
else:
    print(f'Updating hybrid-state-rollback...')
    subprocess.run(['git', '-C', target_dir, 'fetch', 'origin', branch], check=True)
    subprocess.run(['git', '-C', target_dir, 'checkout', branch], check=True)
    subprocess.run(['git', '-C', target_dir, 'pull', 'origin', branch], check=True)

%cd /content/hybrid-state-rollback
!pip install -q -e .
!pip install -q "transformers>=5.0" "datasets>=3.0" accelerate pytest

sys.path.insert(0, '/content/hybrid-state-rollback/src')
os.environ['PYTHONPATH'] = f'/content/hybrid-state-rollback/src:{os.environ.get("PYTHONPATH", "")}'
print('[Ready] hybrid-state-rollback ready on GPU.')


In [ ]:
# Step 3: Run Fast CPU Pre-Flight Checks (Exactness & State-KV Sync)
!pytest -q tests/test_hybrid_rollback.py


In [ ]:
# Step 4: Execute Flagship Option B Kill Experiment (MATH-500 Reasoning Rollback)
# Evaluates reasoning dead-end items across all 7 arms:
# 1. clean (unpolluted ceiling)
# 2. unrolled (polluted CoT)
# 3. attn_kv_only (naive KV crop - exposes state-KV desynchronization)
# 4. prompt_restart (dumb baseline #1)
# 5. full_replay (dumb baseline #2)
# 6. ring_fifo (ReplaySSM baseline)
# 7. sync_binomial (proposed online binomial checkpoint rollback)
!python scripts/run_kill_c1.py run \
    --model Qwen/Qwen3.5-0.8B \
    --dataset math500 \
    --n 100 \
    --pool-cap 4 \
    --device cuda


In [ ]:
# Step 5: Generate Pre-Registered Verdict & Bootstrap 95% Confidence Intervals
import os
if os.path.exists('results/kill_c1.jsonl'):
    !python scripts/run_kill_c1.py report
else:
    print('[Warning] results/kill_c1.jsonl not found. Check Step 4 logs.')


In [ ]:
# Step 6: Credit Guard - Automatically Unassign GPU to Save Credits
print('\n[Credit Guard] Flagship experiment finished! Unassigning Colab GPU immediately...')
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f'Teardown notice: {e}')
